## Reading Bronze.Hotels Delta Table

In [0]:
hotels_bronze_path = "s3://travel-analytics-bronze/delta/bronze/hotels/"
hotels_bronze_df = spark.read.format("delta").load(hotels_bronze_path)

## Silver Transformations

In [0]:
# importing needed functions
from pyspark.sql import functions as F
from pyspark.sql.functions import (
col, trim, upper, to_date, to_timestamp,
    when, date_format, concat, lit, coalesce, expr, initcap
)

# =============================================================
# STEP 0:CONFIGURATION & SETUP for the Destination
# =============================================================
table_name = "hotels"
hotels_silver_path = f"s3://travel-analytics-bronze/delta/silver/{table_name}/"

#=============================================================
# STEP 1: DATA TYPE CASTING & PARSING
#=============================================================
print("\nSTEP 1: Casting Data Types (Hotels)...")

hotels_step_1_df = (
    hotels_bronze_df

    # ========== Numeric columns ==========
    .withColumn("hotel_id", col("hotel_id").cast("int"))
    .withColumn("room_count", col("room_count").cast("int"))
    .withColumn("star_rating", col("star_rating").cast("double"))
    .withColumn("hotel_score", col("hotel_score").cast("double"))

    # ========== String columns ==========
    .withColumn("hotel_name", initcap(trim(col("hotel_name"))))
    .withColumn("hotel_address", initcap(trim(col("hotel_address"))))
    .withColumn("city", initcap(trim(col("city"))))
    .withColumn("country", initcap(trim(col("country"))))

    # ========== CDC updated timestamp ==========
    .withColumn("updated_at", to_timestamp(col("_ab_cdc_updated_at")))
)


# =============================================================
#STEP 2: CLEANING & BUSINESS LOGIC
# =============================================================
print("\nSTEP 2: Cleaning & Standardization (Hotels)...")

hotels_step_2_df = (
    hotels_step_1_df

    # Handle nulls
    .withColumn("hotel_name", coalesce(col("hotel_name"), lit("UNKNOWN")))
    .withColumn("city", coalesce(col("city"), lit("UNKNOWN")))
    .withColumn("country", coalesce(col("country"), lit("UNKNOWN")))

    # Normalize ratings
    .withColumn(
        "star_rating",
        when(col("star_rating").between(0, 5), col("star_rating")).otherwise(None)
    )

    .withColumn(
        "hotel_score",
        when(col("hotel_score").between(0, 10), col("hotel_score")).otherwise(None)
    )
)
# =============================================================
#STEP 3: DEDUPLICATION & DROP AIRBYTE METADATA
# =============================================================
print("\nSTEP 3: Deduplication & Dropping Airbyte Columns (Hotels)...")

airbyte_columns_to_drop = [
    "_airbyte_ab_id",
    "_airbyte_emitted_at",
    "_airbyte_additional_properties",
    "_ab_cdc_lsn",
    "_ab_cdc_deleted_at",
]

hotels_silver_df = (
    hotels_step_2_df

    # Deduplicate on natural key
    .dropDuplicates(["hotel_id"])

    # Drop metadata
    .drop(*airbyte_columns_to_drop)
)




STEP 1: Casting Data Types (Hotels)...

STEP 2: Cleaning & Standardization (Hotels)...

STEP 3: Deduplication & Dropping Airbyte Columns (Hotels)...


In [0]:
#=============================================================
#STEP 4: BUSINESS-FRIENDLY COLUMN NAMES
#=============================================================
print("\nSTEP 4: Renaming Columns (Hotels)...")

rename_map = {
    "hotel_id": "Hotel_Id",
    "hotel_name": "Hotel_Name",
    "hotel_address": "Hotel_Address",
    "city": "City",
    "country": "Country",
    "room_count": "Room_Count",
    "star_rating": "Star_Rating",
    "hotel_score": "Hotel_Score",
    "updated_at": "Updated_At"
}

hotels_silver_df = hotels_silver_df.select(
    [col(c).alias(rename_map.get(c, c)) for c in hotels_silver_df.columns]
)



STEP 4: Renaming Columns (Hotels)...


### Preview: Hotels Silver Table (Cleaned & Deduplicated)

In [0]:
hotels_silver_df.display()

_ab_cdc_updated_at,Hotel_Id,Hotel_Name,Hotel_Address,City,Country,Room_Count,Star_Rating,Hotel_Score,Updated_At
2025-12-12T01:04:52.921893877Z,0,Hotel Arena,55 S Gravesandestraat Oosterparkbuurt Amsterdam North Holland Netherlands 1092aa The Netherlands,Amsterdam,Netherlands,209,3.5,7.7,2025-12-12T01:04:52.921Z
2025-12-12T01:04:52.921893877Z,1,K K Hotel George,Amsterdam Hotel London 7 Trebovir Road Earls Court Royal Borough Of Kensington And Chelsea London Greater London England Sw5 9ls United Kingdom,London,United Kingdom,243,4.5,8.5,2025-12-12T01:04:52.921Z
2025-12-12T01:04:52.921893877Z,2,Apex Temple Court Hotel,162 Fleet Street Temple City Of London Greater London England Ec4 United Kingdom,London,United Kingdom,292,5.0,9.2,2025-12-12T01:04:52.921Z
2025-12-12T01:04:52.921893877Z,3,The Park Grand London Paddington,Park Grand London Paddington 1 Queens Gardens Paddington City Of Westminster London Greater London England W2 3be United Kingdom,London,United Kingdom,310,3.5,7.7,2025-12-12T01:04:52.921Z
2025-12-12T01:04:52.921893877Z,4,Monhotel Lounge Spa,3 Rue Dargentine Quartier De Chaillot 16th Arrondissement Paris Ile De France Metropolitan France 75116 France,Paris,France,246,4.0,8.4,2025-12-12T01:04:52.921Z
2025-12-12T01:04:52.921893877Z,5,Kube Hotel Ice Bar,Kube Hotel 5 Passage Ruelle Quartier De La Goutte Dor 18th Arrondissement Paris Ile De France Metropolitan France 75018 France,Paris,France,139,3.0,7.2,2025-12-12T01:04:52.921Z
2025-12-12T01:04:52.921893877Z,6,The Principal London,Kimpton Fitzroy London Hotel Russell Square Holborn Bloomsbury London Borough Of Camden London Greater London England Wc1h 0lh United Kingdom,London,United Kingdom,257,4.0,8.0,2025-12-12T01:04:52.921Z
2025-12-12T01:04:52.921893877Z,7,Park Plaza County Hall London,Park Plaza 1 Addington Street Lambeth London Borough Of Lambeth London Greater London England Se1 7ry United Kingdom,London,United Kingdom,345,4.0,8.4,2025-12-12T01:04:52.921Z
2025-12-12T01:04:52.921893877Z,8,One Aldwych,One Aldwych 1 Aldwych St Clement Danes Covent Garden City Of Westminster London Greater London England Wc2b 4bz United Kingdom,London,United Kingdom,278,5.0,9.2,2025-12-12T01:04:52.921Z
2025-12-12T01:04:52.921893877Z,9,Splendid Etoile,1 Avenue Carnot Quartier Des Ternes 17th Arrondissement Paris Ile De France Metropolitan France 75017 France,Paris,France,349,4.5,8.9,2025-12-12T01:04:52.921Z


## Writing Silver Hotel_Bookings to Delta Lake with Check-In Date Partitioning

In [0]:
print("\nSTEP 5: Persist Hotels Silver Table...")

hotels_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(hotels_silver_path)



STEP 5: Persist Hotels Silver Table...
